In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.utils import shuffle
from collections import defaultdict
from imblearn.under_sampling import RandomUnderSampler

In [2]:
# Setup all data files as df and sql 
X_train = pd.read_csv("../data/X_train_split.csv")
y_train = pd.read_csv("../data/y_train_split.csv")
X_test = pd.read_csv("../data/X_validation.csv")
y_test = pd.read_csv("../data/y_validation.csv")
X_test_1 = pd.read_csv("../data/X_test_1.csv")
X_test_2 = pd.read_csv("../data/X_test_2.csv")

conn = sqlite3.connect(":memory:")

X_train.to_sql("X_train", conn, index=False, if_exists="replace")
y_train.to_sql("y_train", conn, index=False, if_exists="replace")

8000

Before applying SMOTE TOMEK, data must be pre-processed.

- Standardised
- No outliers
    - when removing outliers, make sure to not remove points from rare classes.
- Preferbly dimension-reduction since the curse of dimensionality makes the euclidian distance redundant for high dimensions.
    - what would be a reasonable dimension?

In [3]:
# Isolation forest to remove potential outliers
iso_models = {}  # save trained iso models per class
X_clean = []
y_clean = []

min_class_count = 50

label_counts = y_train.value_counts()
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.iloc[:, 0]

for label in y_train.unique():
    X_label = X_train[y_train == label]
    y_label = y_train[y_train == label]

    if label_counts[label] >= min_class_count:
        iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
        iso.fit(X_label)
        iso_models[label] = iso  # save model

        preds = iso.predict(X_label)
        X_clean_label = X_label[preds == 1]
        y_clean_label = y_label[preds == 1]
    else:
        X_clean_label = X_label
        y_clean_label = y_label

    X_clean.append(X_clean_label)
    y_clean.append(y_clean_label)

X_clean = pd.concat(X_clean)
y_clean = pd.concat(y_clean)
X_clean, y_clean = shuffle(X_clean, y_clean, random_state=42)

/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/pyth

In [4]:
X_test_clean = []
y_test_clean = []

label_counts_test = y_test.value_counts()
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.iloc[:, 0]

for label in y_test.unique():
    X_label = X_test[y_test == label]
    y_label = y_test[y_test == label]

    if label in iso_models:
        iso = iso_models[label]
        preds = iso.predict(X_label)
        X_label_clean = X_label[preds == 1]
        y_label_clean = y_label[preds == 1]
    else:
        X_label_clean = X_label
        y_label_clean = y_label

    X_test_clean.append(X_label_clean)
    y_test_clean.append(y_label_clean)

X_test_clean = pd.concat(X_test_clean)
y_test_clean = pd.concat(y_test_clean)

In [5]:
# Scale Data and Apply PCA

# Standardize the features (important for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_test_scaled = scaler.fit_transform(X_test_clean)
X_test_1_scaled = scaler.fit_transform(X_test_1)
X_test_2_scaled = scaler.fit_transform(X_test_2)

# Apply PCA to have 50 features
pca = PCA(n_components = 50)
X_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_clean)
X_pca_test_1 = pca.transform(X_test_1_scaled)
X_pca_test_2 = pca.transform(X_test_2_scaled)


total_variance = pca.explained_variance_ratio_.sum()
print(f'Total Variance Explained: {total_variance:.2f}')

Total Variance Explained: 0.74


/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:443: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


In [7]:
X_pca_test_1 = pd.DataFrame(X_pca_test_1)
X_pca_test_2 = pd.DataFrame(X_pca_test_2)

In [9]:
X_pca_test_2

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.863414,-1.684970,-0.519679,1.591336,1.578629,0.124168,0.349354,-3.686097,1.964107,0.044839,...,-1.037588,1.962105,-0.054353,0.110914,-0.950296,0.942159,1.085691,0.404417,-0.203862,-0.222325
1,-4.273566,-3.585091,-1.459164,-1.394476,1.100381,1.052495,-3.685461,-3.381649,0.000059,-3.163575,...,2.034188,-0.590718,1.663430,0.265874,-0.242757,0.830003,0.171378,0.331667,-0.773335,1.929267
2,8.710192,-3.749137,-2.507615,-0.839639,-0.470027,-1.465129,1.688373,-0.490171,0.037954,-3.389179,...,1.712982,-0.193126,0.602661,-1.564261,0.033686,0.073964,0.242097,0.150261,0.771504,-0.565197
3,6.297203,-1.519946,-3.084269,-0.131906,-0.818178,-0.397340,1.926261,0.216742,-0.450755,-2.352555,...,-0.014782,-0.124279,-1.516138,-1.205719,-0.931157,-1.118543,0.513928,1.176502,-0.601494,1.192292
4,14.010882,-0.166471,2.814675,-0.827320,2.240678,-0.985466,0.086910,3.867826,-0.494320,2.370831,...,-0.865720,-1.038638,-1.750258,-0.485955,-0.000036,0.225125,0.182541,-1.019095,-0.536030,0.573491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,-1.272521,-1.088044,7.057967,6.488296,0.994395,3.363668,-0.701995,-2.972719,-3.073925,0.747779,...,0.995883,-0.182405,1.629653,0.355108,1.474729,1.545540,1.364512,-1.026956,1.509286,-0.014898
2016,0.535218,-2.700411,-0.061378,0.062825,-4.930609,-0.157033,-2.112554,-1.828932,-0.751789,-0.163234,...,0.536905,1.593105,-0.540997,-0.374805,1.002455,0.107840,-2.007942,0.207069,-0.376720,0.289324
2017,-0.440616,0.770993,-1.901659,-2.182857,-1.391189,-0.789686,-2.837249,2.624413,0.460969,-2.424161,...,-0.551073,-0.227268,0.027684,-2.072446,-1.172523,1.066360,1.777879,0.169298,-1.900997,-0.106424
2018,-1.415062,2.953084,1.136601,2.099418,2.263928,4.478025,0.759227,-0.454406,8.635347,4.782188,...,-0.616771,2.603236,0.088014,-0.502077,-0.106687,-2.067190,1.088523,1.493010,-0.819458,-1.208644


In [10]:
# Convert X_final (features) to DataFrame
X_test_pca = pd.DataFrame(X_test_pca)
y_test_pca = pd.DataFrame(y_test_clean, columns=['label'])
X_pca = pd.DataFrame(X_pca)
y_pca = pd.DataFrame(y_clean, columns=['label'])

# Export datasets to CSV
X_test_pca.to_csv('../data/X_pca_validation.csv', index=False)
y_test_pca.to_csv('../data/y_pca_validation.csv', index=False)
X_pca.to_csv('../data/X_pca_train.csv', index=False)
y_pca.to_csv('../data/y_pca_train.csv', index=False)


X_pca_test_1 = pd.DataFrame(X_pca_test_1)
X_pca_test_2 = pd.DataFrame(X_pca_test_2)

X_pca_test_1.to_csv('../data/X_pca_test_1.csv', index=False)
X_pca_test_2.to_csv('../data/X_pca_test_2.csv', index=False)

In [10]:
def SMOTE_TOMEK(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 50, 2: 50, 16: 50, 22:50},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            12: 500,   # 457 -> resample to 500
            24: 500,   # 385 -> resample to 500
            17: 500,   # 354 -> resample to 500
            26: 500,   # 278 -> resample to 500
            21: 500,   # 269 -> resample to 500
            14: 500,   # 264 -> resample to 500
            4: 500,    # 238 -> resample to 500
            25: 500,   # 184 -> resample to 500
            19: 500,   # 177 -> resample to 500
            20: 500,   # 153 -> resample to 500
            27: 200,   # 107 -> resample to 200
            7: 200,    # 103 -> resample to 200
            11: 200,   # 78 -> resample to 200
            3: 200,    # 65 -> resample to 200
            13: 200,   # 62 -> resample to 200
            18: 200,   # 60 -> resample to 200
            23: 200,   # 41 -> resample to 200
            15: 200,   # 26 -> resample to 200
            9: 200,    # 25 -> resample to 200
            0: 200     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    return X_tomek, y_tomek

In [11]:
X_final, y_final = SMOTE_TOMEK(X_pca, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
2       50
1       50
16      50
22      50
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
10     856
21     500
19     500
24     500
14     500
4      500
12     500
25     500
20     500
26     500
17     500
6      437
8      408
11     200
0      200
13     200
7      200
23     200
3  

In [14]:
# Export new synthetic dataset

# Convert X_final (features) to DataFrame
X_df = pd.DataFrame(X_final)

# Convert y_final (labels) to DataFrame (with a column name)
y_df = pd.DataFrame(y_final, columns=['label'])

# Export X_df to CSV
X_df.to_csv('../data/X_pca2_SMOTETOMEK.csv', index=False)

# Export y_df to CSV
y_df.to_csv('../data/y_pca2_SMOTETOMEK.csv', index=False)

In [15]:
def SMOTE_TOMEK_UNDER(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 50, 2: 50, 16: 50, 22:50},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            12: 500,   # 457 -> resample to 500
            24: 500,   # 385 -> resample to 500
            17: 500,   # 354 -> resample to 500
            26: 500,   # 278 -> resample to 500
            21: 500,   # 269 -> resample to 500
            14: 500,   # 264 -> resample to 500
            4: 500,    # 238 -> resample to 500
            25: 500,   # 184 -> resample to 500
            19: 500,   # 177 -> resample to 500
            20: 500,   # 153 -> resample to 500
            27: 200,   # 107 -> resample to 200
            7: 200,    # 103 -> resample to 200
            11: 200,   # 78 -> resample to 200
            3: 200,    # 65 -> resample to 200
            13: 200,   # 62 -> resample to 200
            18: 200,   # 60 -> resample to 200
            23: 200,   # 41 -> resample to 200
            15: 200,   # 26 -> resample to 200
            9: 200,    # 25 -> resample to 200
            0: 200     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    rus = RandomUnderSampler(
        sampling_strategy={5:1000},
        random_state=42
    )
    X_under, y_under = rus.fit_resample(X_tomek, y_tomek)
    
    print(f"Label distribution after undersampling class 5:\n{y_under.value_counts()}")
    
    return X_under, y_under

In [16]:
X_under, y_under = SMOTE_TOMEK_UNDER(X_pca, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
2       50
1       50
16      50
22      50
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
10     856
21     500
19     500
24     500
14     500
4      500
12     500
25     500
20     500
26     500
17     500
6      437
8      408
11     200
0      200
13     200
7      200
23     200
3  

In [17]:
# Export new synthetic dataset

# Convert X_final (features) to DataFrame
X_under = pd.DataFrame(X_under)

# Convert y_final (labels) to DataFrame (with a column name)
y_under = pd.DataFrame(y_under, columns=['label'])

# Export X_df to CSV
X_under.to_csv('../data/X_pca2_SMOTETOMEKUNDER.csv', index=False)

# Export y_df to CSV
y_under.to_csv('../data/y_pca2_SMOTETOMEKUNDER.csv', index=False)

In [18]:
# Set all labels to roughly the same number of points of 1000
def balance(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 1000, 2: 1000, 16: 1000, 22:1000},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            10: 1000,
            6: 1000,
            8: 1000,
            12: 1000,   # 457 -> resample to 500
            24: 1000,   # 385 -> resample to 500
            17: 1000,   # 354 -> resample to 500
            26: 1000,   # 278 -> resample to 500
            21: 1000,   # 269 -> resample to 500
            14: 1000,   # 264 -> resample to 500
            4: 1000,    # 238 -> resample to 500
            25: 1000,   # 184 -> resample to 500
            19: 1000,   # 177 -> resample to 500
            20: 1000,   # 153 -> resample to 500
            27: 1000,   # 107 -> resample to 200
            7: 1000,    # 103 -> resample to 200
            11: 1000,   # 78 -> resample to 200
            3: 1000,    # 65 -> resample to 200
            13: 1000,   # 62 -> resample to 200
            18: 1000,   # 60 -> resample to 200
            23: 1000,   # 41 -> resample to 200
            15: 1000,   # 26 -> resample to 200
            9: 1000,    # 25 -> resample to 200
            0: 1000     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    rus = RandomUnderSampler(
        sampling_strategy={5:1000},
        random_state=42
    )
    X_under, y_under = rus.fit_resample(X_tomek, y_tomek)
    
    print(f"Label distribution after undersampling class 5:\n{y_under.value_counts()}")
    
    return X_under, y_under

In [19]:
X_balance, y_balance = balance(X_pca, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
2     1000
16    1000
1     1000
22    1000
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
19    1000
16    1000
1     1000
2     1000
0     1000
13    1000
7     1000
23    1000
11    1000
3     1000
27    1000
15    1000
9     1000
25    1000
24    1000
14    1000
4     1000
6     1000
12 

In [20]:
# Export new synthetic dataset

# Convert X_final (features) to DataFrame
X_balance = pd.DataFrame(X_balance)

# Convert y_final (labels) to DataFrame (with a column name)
y_balance = pd.DataFrame(y_balance, columns=['label'])

# Export X_df to CSV
X_balance.to_csv('../data/X_pca2_balance.csv', index=False)

# Export y_df to CSV
y_balance.to_csv('../data/y_pca2_balance.csv', index=False)